## Comparison of the K-Means Results of all Systems

This notebook compares the K-Means results of CartPole, Pendulum and HalfCheetah in one table. It runs no rollouts, it only loads the saved results of the notebooks in `cartpole/`, `pendulum/` and `halfcheetah/`, so run those first.

- **Prediction strength** (`kmeans_k_selection.ipynb`): the mean prediction strength for k = 2 and k = 3 and the selected k.
- **Prediction error** (`kmeans_repeated_eval.ipynb`): the MAE of the random tasks, compared to the baseline that always predicts 0. The task parameters have very different ranges, so the reduction in percent is compared.
- **Consistency** (`kmeans_consistency.ipynb`): the share of repetitions in which an evenly spaced task lands in its reference cluster.
- **Cluster boundary:** where the clusters are separated, as the middle between the two neighbouring evenly spaced tasks, and how many tasks each cluster contains.

For HalfCheetah, no k reaches the prediction strength threshold (k = 1, no reproducible cluster structure). Its values for k = 2 are for comparison only and marked with *.

In [1]:
import numpy as np
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from analysis_utils import SYSTEMS, CHECKPOINT_DIR, task_parameters

# Display name, checkpoint prefix and task parameter of every system
SYSTEM_INFO = {
    "cartpole": {"name": "CartPole", "prefix": "cartpole_wind_", "parameter": "Wind Magnitude", "limit": 5.0},
    "pendulum": {"name": "Pendulum", "prefix": "pendulum_wind_", "parameter": "Wind Torque", "limit": 0.5},
    "halfcheetah": {"name": "HalfCheetah", "prefix": "halfcheetah_action_", "parameter": "Action Offset", "limit": 0.2},
}

### Load the Results

`selected_k` is the k chosen with the prediction strength (1 if no k reaches the threshold). `used_k` is the k of the evaluation: the selected k, or k = 2 for comparison if no k was selected.

In [2]:
def cluster_boundaries(parameters, labels):
    # Boundaries between neighbouring evenly spaced tasks of different clusters, and the size of every cluster
    order = np.argsort(parameters)
    sorted_parameters, sorted_labels = parameters[order], labels[order]
    changes = np.flatnonzero(sorted_labels[1:] != sorted_labels[:-1])
    boundaries = (sorted_parameters[changes] + sorted_parameters[changes + 1]) / 2
    sizes = np.diff(np.concatenate([[0], changes + 1, [len(parameters)]]))
    return boundaries, sizes


results = {}

for key, info in SYSTEM_INFO.items():
    prefix = info["prefix"]
    k_selection = np.load(CHECKPOINT_DIR / f"{prefix}kmeans_k_selection.npz")
    repeated_eval = np.load(CHECKPOINT_DIR / f"{prefix}kmeans_repeated_eval.npz")
    consistency = np.load(CHECKPOINT_DIR / f"{prefix}kmeans_consistency.npz")

    k_values = [int(k) for k in k_selection["k_values"]]
    mean_ps = k_selection["mean_prediction_strength"]
    selected_k = int(k_selection["best_k"])
    used_k = int(repeated_eval["best_k"])

    # MAE of the random tasks with the used k, averaged over all repetitions
    eval_k_values = [int(k) for k in repeated_eval["k_values"]]
    mae = repeated_eval["mae_results"][:, eval_k_values.index(used_k)].mean()
    baseline = repeated_eval["baseline_maes"].mean()

    # Evenly spaced tasks and their reference clusters
    labels = consistency["reference_cluster_assignments"]
    system = SYSTEMS[key]
    parameters = task_parameters(system, system.sample_tasks_eval(num_envs=len(labels)))
    boundaries, sizes = cluster_boundaries(parameters, labels)

    results[key] = dict(
        selected_k=selected_k,
        used_k=used_k,
        k_values=k_values,
        mean_ps=mean_ps,
        ps_used_k=mean_ps[k_values.index(used_k)],
        mae=mae,
        baseline=baseline,
        reduction=1 - mae / baseline,
        consistency=consistency["agreement_per_repeat"].mean(),
        parameters=parameters,
        labels=labels,
        boundaries=boundaries,
        sizes=sizes,
    )

for key, r in results.items():
    print(f"{SYSTEM_INFO[key]['name']:12s} selected k = {r['selected_k']}, used k = {r['used_k']}")

CartPole     selected k = 2, used k = 2
Pendulum     selected k = 2, used k = 2
HalfCheetah  selected k = 1, used k = 2


### Comparison Table

In [3]:
def mark(key, text):
    # Values of a k below the threshold are marked with *
    return text + ("*" if results[key]["selected_k"] == 1 else "")


def selected_text(key):
    k = results[key]["selected_k"]
    return f"{k} (no cluster structure)" if k == 1 else str(k)


def ps(key, k):
    # Mean prediction strength of a system for one k
    return results[key]["mean_ps"][results[key]["k_values"].index(k)]


def boundary_text(key):
    r = results[key]
    boundaries = ", ".join(f"{b:+.3g}" for b in r["boundaries"])
    sizes = " : ".join(str(s) for s in r["sizes"])
    return mark(key, f"{boundaries} ({sizes} tasks)")


rows = [
    ("Task parameter", [f"{info['parameter']} (±{info['limit']:g})" for info in SYSTEM_INFO.values()]),
    ("Prediction strength k = 2", [f"{ps(key, 2):.2f}" for key in SYSTEM_INFO]),
    ("Prediction strength k = 3", [f"{ps(key, 3):.2f}" for key in SYSTEM_INFO]),
    ("Selected k", [selected_text(key) for key in SYSTEM_INFO]),
    ("Used k", [mark(key, str(results[key]["used_k"])) for key in SYSTEM_INFO]),
    ("MAE / baseline", [mark(key, f"{results[key]['mae']:#.3g} / {results[key]['baseline']:#.3g}") for key in SYSTEM_INFO]),
    ("MAE reduction vs. baseline", [mark(key, f"{results[key]['reduction']:.0%}") for key in SYSTEM_INFO]),
    ("Consistency", [mark(key, f"{results[key]['consistency']:.1%}") for key in SYSTEM_INFO]),
    ("Cluster boundary", [boundary_text(key) for key in SYSTEM_INFO]),
]

label_width = max(len(label) for label, _ in rows)
column_width = max(len(value) for _, values in rows for value in values)
names = [info["name"] for info in SYSTEM_INFO.values()]

print(f"{'':{label_width}} | " + " | ".join(f"{name:{column_width}}" for name in names))
print("-" * (label_width + 3 + len(names) * (column_width + 3) - 3))
for label, values in rows:
    print(f"{label:{label_width}} | " + " | ".join(f"{value:{column_width}}" for value in values))

print("\n* k = 2 for comparison only, no k reaches the prediction strength threshold of 0.8")

                           | CartPole                 | Pendulum                 | HalfCheetah             
-----------------------------------------------------------------------------------------------------------
Task parameter             | Wind Magnitude (±5)      | Wind Torque (±0.5)       | Action Offset (±0.2)    
Prediction strength k = 2  | 0.90                     | 0.95                     | 0.78                    
Prediction strength k = 3  | 0.57                     | 0.74                     | 0.71                    
Selected k                 | 2                        | 2                        | 1 (no cluster structure)
Used k                     | 2                        | 2                        | 2*                      
MAE / baseline             | 1.42 / 2.60              | 0.128 / 0.256            | 0.0500 / 0.102*         
MAE reduction vs. baseline | 45%                      | 50%                      | 51%*                    
Consistency                |

The prediction strength of every tested k, for all systems. The selected k is marked with `<`.

In [4]:
print(f"{'k':>3} | " + " | ".join(f"{info['name']:>12}" for info in SYSTEM_INFO.values()))

for j, k in enumerate(results["cartpole"]["k_values"]):
    values = []
    for key in SYSTEM_INFO:
        marker = " <" if k == results[key]["selected_k"] else "  "
        values.append(f"{results[key]['mean_ps'][j]:>10.3f}{marker}")
    print(f"{k:>3} | " + " | ".join(values))

  k |     CartPole |     Pendulum |  HalfCheetah
  2 |      0.901 < |      0.950 < |      0.781  
  3 |      0.572   |      0.737   |      0.710  
  4 |      0.519   |      0.682   |      0.496  
  5 |      0.421   |      0.279   |      0.368  
  6 |      0.320   |      0.165   |      0.277  
  7 |      0.192   |      0.117   |      0.237  
  8 |      0.125   |      0.083   |      0.183  
  9 |      0.100   |      0.083   |      0.183  
 10 |      0.120   |      0.017   |      0.092  
